# Озеро ODS: март-дубли INT и май-объём trx

Две **независимые** проверки. Не смешивать в одном выводе озерщикам.

| # | Проблема | Таблица | Что должно получиться |
|---|---|---|---|
| 1 | Дубли INT-комиссии только в марте | `ods_alpha.scd1_trx_int` (`n_amt_fee`) | март ≈ **667k** `n_trx` с 2 одинаковыми живыми строками; фев/апр ≈ 0 |
| 2 | Просадка trx в мае | `ods_alpha.scd1_trx` (+ `trx_acq`, FIID) | тоталы `trx_cnt` / `trx_sum` апр–май–июн |

**55% vs 98%** — это сверка Excel vs `final_df` в боевой тетрадке, не SQL. Здесь проверяем, просел ли **сам ODS** в мае. Если май в озере как апрель/июнь, 55% скорее в Excel/сверке.

Гонять ячейки сверху вниз. `MEM_LIMIT=16g`.


In [ ]:
from IPython.display import display
from rail_connectors.connection import connect

MEM_LIMIT = '16g'

if 'imp' not in globals() or imp is None:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'},
    )
    imp._init_connection()
    print('Impala connected')
else:
    print('Reuse existing imp')


def run_sql(sql, title=None):
    if title:
        print(title)
    with imp:
        imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
        df = imp.fetch(sql)
    display(df)
    return df


## 1) Март: дубли строк `n_amt_fee` в `scd1_trx_int`

Живые строки (`ods_deleted_flg <> 1`), больше одной на `n_trx`, **один и тот же** `n_amt_fee`.

Месяц берём из `scd1_trx.d_trx_orig` (как витрина). Фильтры trx упрощённые: SA / S01 / не reversed / не deleted. Без RSHB и agreements — чтобы запрос был короче; число может быть чуть выше 666 739.

Ожидание: **2026-03** ≈ 667k `same_fee_dups`, avg rows = 2; **2026-02** и **2026-04** ≈ 0.


In [ ]:
sql_march_dups_by_month = r"""
with trx_m as (
  select
    cast(t.n_trx as string) as n_trx,
    trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM') as trx_month
  from ods_alpha.scd1_trx t
  where coalesce(t.ods_deleted_flg, '0') <> '1'
    and t.c_trx_class = 'SA'
    and t.c_trx_type = 'S01'
    and t.c_nter is not null
    and coalesce(t.cf_trx_stat, '') <> 'R'
    and trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM')
        between date '2026-02-01' and date '2026-04-01'
),
int_per_trx as (
  select
    cast(i.n_trx as string) as n_trx,
    count(*) as int_rows,
    count(distinct i.n_amt_fee) as distinct_fee,
    sum(cast(i.n_amt_fee as double)) as fee_sum,
    max(cast(i.n_amt_fee as double)) as fee_max
  from ods_alpha.scd1_trx_int i
  join trx_m t on t.n_trx = cast(i.n_trx as string)
  where coalesce(i.ods_deleted_flg, '0') <> '1'
  group by cast(i.n_trx as string)
)
select
  t.trx_month,
  count(*) as trx_with_int,
  sum(case when i.int_rows > 1 then 1 else 0 end) as multi_trx,
  sum(case when i.int_rows > 1 and i.distinct_fee = 1 then 1 else 0 end) as same_fee_dups,
  sum(case when i.int_rows > 1 and i.distinct_fee > 1 then 1 else 0 end) as different_fee,
  avg(case when i.int_rows > 1 then i.int_rows end) as avg_rows_on_multi,
  sum(case when i.int_rows > 1 and i.distinct_fee = 1
           then i.fee_sum - i.fee_max else 0 end) as extra_fee_sum_minus_max
from trx_m t
join int_per_trx i on i.n_trx = t.n_trx
group by t.trx_month
order by 1
"""

march_by_month = run_sql(
    sql_march_dups_by_month,
    '=== 1. Дубли trx_int по месяцам (фев / март / апр) ===',
)


Пример двух живых строк на один `n_trx` (март). Ожидание: одинаковый `n_amt_fee`, `ods_deleted_flg=0`, часто `ods_op_type='SQL COMPUPDATE'`.


In [ ]:
sql_march_sample = r"""
with trx_mar as (
  select cast(t.n_trx as string) as n_trx
  from ods_alpha.scd1_trx t
  where coalesce(t.ods_deleted_flg, '0') <> '1'
    and t.c_trx_class = 'SA'
    and t.c_trx_type = 'S01'
    and t.c_nter is not null
    and coalesce(t.cf_trx_stat, '') <> 'R'
    and trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM') = date '2026-03-01'
),
multi as (
  select cast(i.n_trx as string) as n_trx
  from ods_alpha.scd1_trx_int i
  join trx_mar t on t.n_trx = cast(i.n_trx as string)
  where coalesce(i.ods_deleted_flg, '0') <> '1'
  group by cast(i.n_trx as string)
  having count(*) > 1 and count(distinct i.n_amt_fee) = 1
  limit 8
)
select
  cast(i.n_trx as string) as n_trx,
  i.n_amt_fee,
  i.ods_deleted_flg,
  i.ods_op_type,
  i.d_trx,
  i.d_int_setl,
  i.d_net_setl
from ods_alpha.scd1_trx_int i
join multi m on m.n_trx = cast(i.n_trx as string)
where coalesce(i.ods_deleted_flg, '0') <> '1'
order by n_trx, i.n_amt_fee
"""

march_sample = run_sql(sql_march_sample, '=== 1b. Sample дублей (март) ===')


## 2) Май: количество и сумма транзакций в ODS

`trx_cnt` = `count(distinct n_trx)`, `trx_sum` = `sum(n_amt_src)`.
Фильтры как в section 05 витрины, без договоров.

Смотрим **апрель / май / июнь**. Если май в озере не просел, а сверка с Excel 55% — копать Excel. Если май просел уже здесь — это озеро (`scd1_trx`).


In [ ]:
sql_may_trx_simple = r"""
select
  trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM') as trx_month,
  count(*) as rows_cnt,
  count(distinct t.n_trx) as trx_cnt,
  sum(cast(t.n_amt_src as double)) as trx_sum
from ods_alpha.scd1_trx t
where coalesce(t.ods_deleted_flg, '0') <> '1'
  and t.c_trx_class = 'SA'
  and t.c_trx_type = 'S01'
  and t.c_nter is not null
  and coalesce(t.cf_trx_stat, '') <> 'R'
  and trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM')
      between date '2026-04-01' and date '2026-06-01'
group by 1
order by 1
"""

may_simple = run_sql(
    sql_may_trx_simple,
    '=== 2. ODS trx_cnt / trx_sum (только scd1_trx) ===',
)


Тот же объём, но как в витрине: только RSHB + есть строка в `scd1_trx_acq`. Если здесь май просел сильнее, чем в запросе выше — дыра в `trx_acq` или в справочнике FIID.


In [ ]:
sql_may_trx_mart = r"""
with fiid_rshb as (
  select distinct cast(fa.c_fiid as string) as c_fiid
  from ods_alpha.scd1_base24_fiids fa
  where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
),
trx_base as (
  select
    cast(t.n_trx as string) as n_trx,
    cast(t.n_amt_src as double) as n_amt_src,
    trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM') as trx_month
  from ods_alpha.scd1_trx t
  join fiid_rshb fr on fr.c_fiid = cast(t.c_fiid_acq as string)
  where coalesce(t.ods_deleted_flg, '0') <> '1'
    and t.c_trx_class = 'SA'
    and t.c_trx_type = 'S01'
    and t.c_nter is not null
    and coalesce(t.cf_trx_stat, '') <> 'R'
    and trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM')
        between date '2026-04-01' and date '2026-06-01'
)
select
  tb.trx_month,
  count(distinct tb.n_trx) as trx_cnt,
  sum(tb.n_amt_src) as trx_sum,
  count(distinct case when a.n_trx is null then tb.n_trx end) as trx_without_acq
from trx_base tb
left join (
  select distinct cast(n_trx as string) as n_trx
  from ods_alpha.scd1_trx_acq
  where coalesce(ods_deleted_flg, '0') <> '1'
) a on a.n_trx = tb.n_trx
group by tb.trx_month
order by 1
"""

may_mart = run_sql(
    sql_may_trx_mart,
    '=== 2b. Как витрина: RSHB + trx_acq (сироты отдельно) ===',
)


Посуточно: нет ли дыр в конце мая (обрыв загрузки).


In [ ]:
sql_may_daily = r"""
select
  to_date(cast(t.d_trx_orig as timestamp)) as trx_dt,
  count(distinct t.n_trx) as trx_cnt,
  sum(cast(t.n_amt_src as double)) as trx_sum
from ods_alpha.scd1_trx t
where coalesce(t.ods_deleted_flg, '0') <> '1'
  and t.c_trx_class = 'SA'
  and t.c_trx_type = 'S01'
  and t.c_nter is not null
  and coalesce(t.cf_trx_stat, '') <> 'R'
  and trunc(to_date(cast(t.d_trx_orig as timestamp)), 'MM') = date '2026-05-01'
group by 1
order by 1
"""

may_daily = run_sql(sql_may_daily, '=== 2c. Май по дням ===')
if may_daily is not None and len(may_daily):
    print('дней с данными:', len(may_daily), '| min', may_daily['trx_dt'].min(), '| max', may_daily['trx_dt'].max())


## Что отправить озерщикам

**Тикет 1 — март.** Таблица `ods_alpha.scd1_trx_int`. Симптом: у сотен тысяч `n_trx` две живые строки с одним `n_amt_fee`. Контроль — февраль и апрель. SQL — ячейка «1. Дубли по месяцам» + sample.

**Тикет 2 — май.** Таблицы `ods_alpha.scd1_trx`, при сиротах ещё `scd1_trx_acq` и `scd1_base24_fiids`. SQL — ячейки 2 и 2b. Цифру 55% в тикет не ставить: её озерщики не воспроизведут без Excel.
